# Sesión 3 — Tuberías de aprendizaje automático (Parte I)
### Definición de problemas · Ingestión de datos · Preparación de datos
**Inteligencia Artificial (FP13)**

---

En esta práctica, trabajamos las **primeras tres etapas** del *machine learning pipeline* usando un
dataset real de la industria automotriz: registros históricos de **powertrain y eficiencia de
combustible** (1970–1982, dataset *Auto MPG* de UCI / Kaggle).

| Etapa del pipeline | Tema del temario | Estado |
|---|---|---|
| 1. Definición del problema | **3.1** | ✅ Esta práctica |
| 2. Ingestión de datos | **3.2** | ✅ Esta práctica |
| 3. Preparación de datos | **3.3** | ✅ Esta práctica |
| 4. Segregación de datos | 3.4 | Siguiente práctica |
| 5. Modelo de entrenamiento | 3.5 | Siguiente práctica |

<br>

**Nota de alcance:** la *normalización* y la *codificación one-hot* se presentan de forma
**superficial** (vista previa). Se profundizan en la **Unidad 4** (Selección e ingeniería de funciones).


> ### Cómo trabajar este notebook
> Este cuaderno está **vacío a propósito**. Cada celda de código trae un marcador
> `# >>> BLOQUE n <<<`. Localiza en el cuadernillo el bloque con el **mismo número**,
> cópialo, pégalo **debajo del marcador** y ejecuta la celda con `Shift + Enter`.
>
> El notebook debe guardarse en la misma carpeta que contiene el directorio `Dataset_1/`
> con los cuatro archivos de datos.


## 0 · Preparación del entorno
Importamos las librerías y fijamos opciones de visualización.

In [ ]:
# >>> BLOQUE 1 <<<  Importación de librerías
# Copia aquí el código del cuadernillo y ejecuta con Shift + Enter



## 3.1 · Definición del problema
> *"Bob dedicó años a planear, ejecutar y optimizar cómo conquistar una colina.
> Por desgracia, resultó ser la colina equivocada."*

Antes de tocar una sola línea de datos hay que **enmarcar la pregunta correcta**. Una mala
definición cuesta órdenes de magnitud más adelante en el pipeline.

**Contexto de negocio (automotriz).** Un fabricante quiere entender qué características del
*powertrain* determinan la **eficiencia de combustible** de un vehículo, para orientar el diseño
de futuras plataformas.

| Iteración | Pregunta | Problema |
|---|---|---|
| v1 | ¿El auto es eficiente, sí o no? | Binario pobre; pierde magnitud |
| v2 | ¿Cuántas millas por galón rinde? | Mejor, pero unidad imperial |
| v3 | ¿Cuál es el **consumo (L/100 km)** dado el powertrain? | Métrica útil para ingeniería |
| v4 | ¿Consumo **sin usar variables no permitidas** (p. ej. el nombre comercial)? | Evita fugas y sesgos |

**Problema definido para esta sesión:** *predecir el consumo (L/100 km) de un vehículo a
partir de sus características técnicas (cilindros, cilindrada, potencia, peso, etc.).*
Esta definición determina **qué datos** ingerimos y **cómo** los preparamos.


## 3.2 · Ingestión de datos
Los datos casi nunca llegan en un solo archivo ni en un solo formato. En este caso el proveedor
entrega **4 fuentes** en **4 formatos** distintos:

| Fuente | Archivo | Formato | Detalle a cuidar |
|---|---|---|---|
| Planta Norteamérica | `flota_norteamerica.csv` | CSV | Separador por comas |
| Planta Europa | `flota_europa.xlsx` | Excel | Varias hojas (`sheet_name`) |
| Planta Asia | `flota_asia.txt` | TXT delimitado por `~` | Valores faltantes |
| Data lake | `flota_datalake.parquet` | Parquet | Formato columnar binario |


### 3.2.1 · CSV — `pd.read_csv`

In [ ]:
# >>> BLOQUE 2 <<<  Lectura del archivo CSV
# Copia aquí el código del cuadernillo y ejecuta con Shift + Enter



### 3.2.2 · Excel — `pd.read_excel`
Un `.xlsx` puede tener varias hojas. Elegimos la hoja de datos con `sheet_name`.
La segunda hoja es un **diccionario de datos** (muy útil para documentar el proyecto).

In [ ]:
# >>> BLOQUE 3 <<<  Hojas del archivo y lectura de datos
# Copia aquí el código del cuadernillo y ejecuta con Shift + Enter



In [ ]:
# >>> BLOQUE 4 <<<  Lectura del diccionario de datos
# Copia aquí el código del cuadernillo y ejecuta con Shift + Enter



### 3.2.3 · TXT delimitado por `~` — `pd.read_csv` con `sep`
`read_csv` sirve para **cualquier** texto delimitado, no solo comas. Aquí el separador es `~`.

In [ ]:
# >>> BLOQUE 5 <<<  Lectura con separador personalizado
# Copia aquí el código del cuadernillo y ejecuta con Shift + Enter



### 3.2.4 · Parquet — `pd.read_parquet`
Parquet es un formato **columnar y comprimido**, el estándar en *data lakes* (Spark, Hadoop,
S3). Conserva los tipos de dato y ocupa mucho menos espacio que un CSV.

In [ ]:
# >>> BLOQUE 6 <<<  Lectura del formato columnar
# Copia aquí el código del cuadernillo y ejecuta con Shift + Enter



### 3.2.5 · Consolidación de fuentes
Como las cuatro fuentes comparten esquema, las **apilamos** con `pd.concat`. En un proyecto real
podríamos necesitar `merge`/`join` si vinieran con llaves distintas.

> Otros formatos frecuentes que se leen igual de fácil: `pd.read_json`, `pd.read_sql`,
> `pd.read_html`.

In [ ]:
# >>> BLOQUE 7 <<<  Apilado de los cuatro DataFrames
# Copia aquí el código del cuadernillo y ejecuta con Shift + Enter



## 3.3 · Preparación de datos
Es la etapa que **más tiempo consume** en ciencia de datos. Recorremos las siete técnicas del
libro, en el orden del temario.

### 3.3.0 · Radiografía inicial
Antes de limpiar, hay que *entender* el estado de los datos.

In [ ]:
# >>> BLOQUE 8 <<<  Estructura y tipos de dato
# Copia aquí el código del cuadernillo y ejecuta con Shift + Enter



In [ ]:
# >>> BLOQUE 9 <<<  Duplicados y valores faltantes
# Copia aquí el código del cuadernillo y ejecuta con Shift + Enter



In [ ]:
# >>> BLOQUE 10 <<<  Estadística descriptiva
# Copia aquí el código del cuadernillo y ejecuta con Shift + Enter



### (1) Imputar valores faltantes
`horsepower` tiene faltantes. El libro describe **seis estrategias**:

1. **No hacer nada** (algoritmos como XGBoost los toleran).
2. **Imputar con la mediana** — rápido y robusto a valores extremos ✅ *(la usaremos)*.
3. **Imputar con la media** — sensible a outliers.
4. **Valor más frecuente / constante** — sirve para variables no numéricas.
5. **Imputación por modelo** (KNN, regresión) — más precisa, más costosa.
6. **Eliminar las filas/columnas** — solo si son muy pocas.

Usamos la **mediana** porque la potencia tiene distribución sesgada y algunos valores altos.

In [ ]:
# >>> BLOQUE 11 <<<  Imputación por mediana en horsepower
# Copia aquí el código del cuadernillo y ejecuta con Shift + Enter



### (2) Eliminar registros duplicados
El *data lake* reingirió registros que ya existían en otras fuentes. `drop_duplicates` elimina
filas idénticas.

> En la práctica, los duplicados "difusos" (mismo auto con pequeñas diferencias de texto) no se
> detectan con igualdad exacta; ahí se usa **fuzzy matching**.

In [ ]:
# >>> BLOQUE 12 <<<  drop_duplicates con reindexado
# Copia aquí el código del cuadernillo y ejecuta con Shift + Enter



### (3) Normalización y codificación one-hot — *vista previa*
> ⚠️ **Superficial.** Se profundiza en la **Unidad 4**. Aquí solo mostramos qué hacen.

**Normalización / escalado:** lleva variables de distintas magnitudes a una escala común. Muchos
algoritmos que usan distancias (KNN, SVM) lo necesitan.

In [ ]:
# >>> BLOQUE 13 <<<  Estandarización Z-score y Min-Max
# Copia aquí el código del cuadernillo y ejecuta con Shift + Enter



**Codificación one-hot:** convierte una variable categórica en columnas binarias (0/1), porque
los modelos no operan con texto. Ejemplo con la región de origen.

In [ ]:
# >>> BLOQUE 14 <<<  Codificación one-hot de la región
# Copia aquí el código del cuadernillo y ejecuta con Shift + Enter



### (4) Otro tipo de limpieza y mapeos
Tres tareas típicas:
1. **Renombrar columnas** a español (legibilidad del proyecto).
2. **Mapear códigos** de `origin` (1/2/3) a etiquetas de región.
3. **Convertir unidades imperiales a métricas** — clave para ingeniería automotriz en México
   (litros, kilogramos, L/100 km).

In [ ]:
# >>> BLOQUE 15 <<<  Renombrado, mapeo y conversión de unidades
# Copia aquí el código del cuadernillo y ejecuta con Shift + Enter



### (5) Extracción de características
La columna `nombre` es **texto libre** que mezcla marca y modelo. Extraemos la **marca** (primer
token) y, de paso, corregimos inconsistencias reales del dataset (errores de captura y alias):
`chevy`/`chevroelt` → `chevrolet`, `maxda` → `mazda`, `toyouta` → `toyota`,
`vw`/`vokswagen` → `volkswagen`.

In [ ]:
# >>> BLOQUE 16 <<<  Extracción y normalización de la marca
# Copia aquí el código del cuadernillo y ejecuta con Shift + Enter



In [ ]:
# Explicación del código de normalización de marcas

# ==============================================================================
# EXPLICACIÓN PASO A PASO:
# df['marca'] = df['nombre'].str.split().str[0].replace(correcciones)
# ==============================================================================
#
# 1. df['nombre']
#    Toma la columna original que contiene el nombre completo del auto.
#    Ejemplo: "chevroelt malibu lz"
#
# 2. .str.split()
#    Trata el texto como cadena (str) y lo divide por los espacios en blanco, 
#    creando una lista de palabras por cada fila.
#    Ejemplo: ["chevroelt", "malibu", "lz"]
#
# 3. .str[0]
#    Accede a la lista recién creada y extrae únicamente el primer elemento 
#    (el índice 0), asumiendo que la primera palabra siempre es la marca.
#    Ejemplo: "chevroelt"
#
# 4. .replace(correcciones)
#    Toma esa primera palabra y la busca en el diccionario de correcciones.
#    Si la encuentra (ej. "chevroelt"), la cambia por su valor ("chevrolet").
#    Si NO la encuentra (ej. "ford"), la deja intacta.
#
# 5. df['marca'] = ...
#    Asigna todo este resultado final a una nueva columna llamada 'marca'.
# ==============================================================================

### (6) Eliminar variables correlacionadas
Dos variables muy correlacionadas aportan **información redundante** (multicolinealidad):
inflan la complejidad, desestabilizan algunos modelos y dificultan la interpretación.

In [ ]:
# >>> BLOQUE 18 <<<  Matriz de correlación
# Copia aquí el código del cuadernillo y ejecuta con Shift + Enter



**Mapa de calor de correlaciones** — el recurso visual clave de esta etapa.

In [ ]:
# >>> BLOQUE 19 <<<  Mapa de calor con máscara triangular
# Copia aquí el código del cuadernillo y ejecuta con Shift + Enter



**Matriz de dispersión** — muestra las relaciones par a par de forma cruda.

In [ ]:
# >>> BLOQUE 20 <<<  Matriz de dispersión (pairplot)
# Copia aquí el código del cuadernillo y ejecuta con Shift + Enter



**Detección automática de pares redundantes** (`|r| > 0.9`) y decisión de qué conservar.

In [ ]:
# >>> BLOQUE 21 <<<  Detección automática de pares redundantes
# Copia aquí el código del cuadernillo y ejecuta con Shift + Enter



**Criterio de la clase:** `cilindros`, `cilindrada_L` y `peso_kg` describen "el tamaño del
motor/auto". Conservamos **`peso_kg`** (la más informativa para consumo) y descartamos
`cilindrada_L` y `cilindros` como candidatas redundantes.

In [ ]:
# >>> BLOQUE 22 <<<  Eliminación de columnas redundantes
# Copia aquí el código del cuadernillo y ejecuta con Shift + Enter



### (7) Ingeniería de características
Creamos variables **nuevas y más predictivas** combinando las existentes. Todas con sentido
físico/automotriz:

- `relacion_pot_peso` = potencia / peso → **relación potencia-peso** (desempeño).
- `cilindrada_por_cil` = cilindrada / cilindros → **cilindrada unitaria**.
- `antiguedad` = 82 − año de modelo → posición temporal.


In [ ]:
# >>> BLOQUE 23 <<<  Creación de variables derivadas
# Copia aquí el código del cuadernillo y ejecuta con Shift + Enter



## Cierre — dataset listo para modelar
Partimos de 4 archivos crudos en 4 formatos y obtuvimos un dataset **consolidado, limpio y
enriquecido**.


In [ ]:
# >>> BLOQUE 24 <<<  Descarte de unidades imperiales
# Copia aquí el código del cuadernillo y ejecuta con Shift + Enter



In [ ]:
# >>> BLOQUE 25 <<<  Verificación final del dataset
# Copia aquí el código del cuadernillo y ejecuta con Shift + Enter

